# PCW Lesson 5: Naive Bayes - SMS Spam Classification

**Student**: Katia Gwaneza Nkurunziza  
**Date**: Session 5, Fall 2026  
**Topics**: Naive Bayes, TF-IDF, joint vs marginal probability, generative classification

**Note on dataset URL**: the original dataset link in the PCW (`milindsoorya/SpamClassifier-in-python`) returns a 404, the GitHub repo no longer exists. I found a working mirror of the same classic SMS Spam Collection dataset (`mohitgupta-omg/Kaggle-SMS-Spam-Collection-Dataset-`). Confirmed it's the same dataset the PCW intends: after vectorizing, it produces exactly **8672 features**, matching the number stated in Question 4 of the assignment.

## LLM Prompt Used

> I am an undergraduate computer science student, I'm trying to build intuition for Naive Bayes classification before class. I'd like you to pretend to be a machine learning tutor and explain Naive Bayes, including how it uses Bayes' theorem and why it's called "naive." Please format your answer as a short list of key ideas with one example each.

Follow-up redirects used:
> Thanks, that's close. Here's some additional context: I already understand basic conditional probability and Bayes' theorem from Session 1's extension problems. Can you focus specifically on what the "naive" independence assumption means mathematically, and give a concrete case where that assumption is clearly false?

## Question 1 of 7: Basic Questions

### 1. What makes naive Bayes "naive"?

Naive Bayes assumes every feature is **conditionally independent of every other feature, given the class**. In plain terms: once you know a message is "spam," knowing the word "free" appears tells you nothing extra about whether the word "win" also appears. Mathematically:

```
P(x_1, x_2, ..., x_n | class) = P(x_1 | class) * P(x_2 | class) * ... * P(x_n | class)
```

This is "naive" because real features are almost never truly independent. Words in real sentences depend on each other (grammar, topic, style). The model just pretends they don't, because it makes the math tractable: instead of needing the full joint distribution over all word combinations (which would need an astronomical number of parameters), you only need one probability per word per class.

### 2. What's a latent variable?

A latent variable is a variable that influences the data you observe, but that you never directly measure. In spam classification, "is this message spam?" is kind of like a latent cause: it's not written anywhere in the text itself, but it's the hidden factor that shapes which words are more or less likely to appear. The model tries to infer this hidden label from the words it can observe.

### 3. What's a case where the naive assumption breaks down?

Phrases where words are strongly linked: "not bad" versus "bad." If you treat "not" and "bad" as independent, the model can't tell that "not bad" is actually mildly positive, it just sees two words that individually lean negative. Another example: "free" and "prize" showing up together is a much stronger spam signal than either word alone (they co-occur constantly in spam), but naive Bayes can't capture that extra co-occurrence boost, it just multiplies their individual spam-probabilities as if seeing one told you nothing about the other.

### 4. Explain the difference between a joint probability and marginal probability.

**Joint probability** `P(A, B)`: the probability that both A and B happen together. Example: P(message contains "free" AND message is spam).

**Marginal probability** `P(A)`: the probability of A happening on its own, ignoring B entirely, obtained by summing (or integrating) the joint probability over all possible values of B. Example: P(message is spam), regardless of what words it contains.

"Marginal" comes from the historical practice of writing these sums in the margins of a probability table.

### 5. When would the joint probability not equal the product of the marginals?

`P(A, B) = P(A) * P(B)` only holds when A and B are **independent**. Whenever A and B are related (dependent), the joint probability differs from that product. Example: P(word="free") on its own might be 2%, and P(spam) on its own might be 13%. But P(word="free" AND spam) is much higher than `0.02 * 0.13 = 0.0026`, because "free" is disproportionately common in spam specifically. That gap between the true joint and the independence-assumed product is exactly the thing naive Bayes chooses to ignore.

## Core Question: SMS Classification

Building a spam/ham classifier for SMS text messages.

In [ ]:
# load in dataset
import pandas as pd
from sklearn.feature_extraction import text

# Original PCW URL (milindsoorya/SpamClassifier-in-python) returns 404 - repo no longer exists.
# Using a mirror of the same classic SMS Spam Collection dataset instead.
df = pd.read_csv("https://raw.githubusercontent.com/mohitgupta-omg/Kaggle-SMS-Spam-Collection-Dataset-/master/spam.csv", encoding='latin-1')
labels, texts = df['v1'], df['v2']
vectorizer = text.TfidfVectorizer()
vec_texts = vectorizer.fit_transform(texts)

print(f"Dataset shape: {df.shape}")
print(f"Label counts:\n{labels.value_counts()}")
print(f"\nVectorized shape: {vec_texts.shape}")
print(f"Vocabulary size: {len(vectorizer.vocabulary_)}")

# Uncomment and print one at a time
print("\nFirst 3 messages:")
print(texts[:3])
# Hint: vec_texts is a sparse matrix; the coordinates are only shown for its "nonzero" entries
print("\nFirst 3 rows of vec_texts (sparse):")
print(vec_texts[:3], vec_texts[:3].shape)

## Question 2 of 7: What is TfidfVectorizer Doing?

**TfidfVectorizer** converts each text message into a numeric vector, one number per unique word in the whole dataset's vocabulary (here, 8672 words), so it can be fed into a machine learning model. Each number is a **TF-IDF score**: Term Frequency times Inverse Document Frequency.

**Term Frequency (TF)** measures how often a word appears in *this specific message*: words that show up a lot in one message get a higher score for that message.

**Inverse Document Frequency (IDF)** measures how *rare* a word is across *all* messages: common words like "the" or "you" that appear in almost every message get down-weighted, while rare, more distinctive words (like "prize" or "txt") get up-weighted.

Multiplying TF by IDF means a word only scores highly if it appears often in this particular message AND is relatively rare overall, exactly the kind of word that's actually informative about what a specific message is about, rather than just a common filler word that appears everywhere. The result is a sparse vector (mostly zeros, since any one message only uses a tiny fraction of the full vocabulary).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import metrics, naive_bayes

# We train a model
model = naive_bayes.MultinomialNB()
model.fit(vec_texts, labels)
pred_labels = model.predict(vec_texts)

confusion_mat = metrics.confusion_matrix(labels, pred_labels)
sns.heatmap(confusion_mat, square=True, annot=True, fmt='d', cbar=False,
            xticklabels=model.classes_, yticklabels=model.classes_)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

print(f"Classes order: {model.classes_}")
print(f"Confusion matrix:\n{confusion_mat}")
print(f"\nAccuracy: {metrics.accuracy_score(labels, pred_labels):.4f}")
print(f"\n{metrics.classification_report(labels, pred_labels, digits=4)}")

## Question 3 of 7: How Well Does This Model Perform?

From the confusion matrix (rows = true label, columns = predicted label, classes ordered `[ham, spam]`):

```
              Predicted ham   Predicted spam
True ham           4825              0
True spam           132            615
```

**Overall accuracy: 97.63%**

But accuracy alone hides an important asymmetry, since the classes are imbalanced (4825 ham vs 747 spam):

- **Ham recall = 100%**: every single real ham message was correctly labeled ham. Zero false positives (no ham was ever mistakenly flagged as spam).
- **Spam recall = 82.3%** (615 out of 747): the model misses about 1 in 6 spam messages, letting them through as "ham."
- **Spam precision = 100%**: every message the model *did* flag as spam was actually spam. It never cries wolf.

**In plain terms**: this model is conservative. It never annoys you by blocking a real message (perfect ham recall, perfect spam precision), but it lets a meaningful chunk of spam slip through undetected (82% spam recall). For a spam filter, that's actually a reasonable tradeoff, missing some spam is far less costly than accidentally hiding a real text from a friend.

## Question 4 of 7: How Many Total Parameters?

**Setup**:
- Naive Bayes learns one probability per feature, per class, independently, then multiplies them together (the naive assumption from Question 1)
- TF-IDF vectors are 8672-dimensional (8672 unique words in the vocabulary)
- There are 2 classes: spam and ham

**Parameters needed**:

1. **Per-feature, per-class likelihoods**: for every one of the 8672 words, the model needs to learn P(word | spam) and P(word | ham) separately. That's:
```
8672 words * 2 classes = 17344 parameters
```

2. **Class priors**: the model also needs P(spam) and P(ham), the baseline probability of each class before looking at any words:
```
2 classes = 2 parameters
```

**Total: 17344 + 2 = 17,346 parameters**

**Verified numerically** using the actual fitted model below: `model.feature_log_prob_` has shape `(2, 8672)` = 17,344 values, and `model.class_log_prior_` has shape `(2,)` = 2 values. Total matches: 17,346.

In [ ]:
n_features = vec_texts.shape[1]
n_classes = len(model.classes_)

print(f"n_features: {n_features}")
print(f"n_classes: {n_classes}")
print(f"feature_log_prob_ shape: {model.feature_log_prob_.shape}  -> {model.feature_log_prob_.size} parameters")
print(f"class_log_prior_ shape: {model.class_log_prior_.shape}  -> {model.class_log_prior_.size} parameters")

total_params = model.feature_log_prob_.size + model.class_log_prior_.size
print(f"\nTotal parameters: {total_params}")
print(f"Matches hand calculation (8672*2 + 2 = {8672*2 + 2}): {total_params == 8672*2 + 2}")

## Question 5 of 7: How Does Naive Bayes Divide Data Geometrically?

In Session 4, decision trees divided the feature space into **axis-aligned rectangles**, carving up 2D space with a staircase of horizontal and vertical cuts, one feature examined at a time.

Naive Bayes does something fundamentally different: there is no explicit "tree" of yes/no questions being fit at all. Instead, for a new message, the model computes:

```
P(spam | words) is proportional to P(spam) * P(word_1 | spam) * P(word_2 | spam) * ...
P(ham | words)  is proportional to P(ham)  * P(word_1 | ham)  * P(word_2 | ham)  * ...
```

and predicts whichever class has the higher score. Geometrically (thinking of each message as a point in 8672-dimensional word-count space), this comparison of two products of probabilities works out to be equivalent to a **single straight (linear) decision boundary** through that high-dimensional space, a hyperplane, not a staircase of boxes.

**Why linear?** Taking the log of both sides turns each product into a *sum*:
```
log P(spam | words) = log P(spam) + sum_i [ log P(word_i | spam) ] 
```
This is a weighted sum of features (each word contributes an additive log-probability weight), exactly the same mathematical shape as `w_1*x_1 + w_2*x_2 + ... + b` from linear/logistic regression (Session 2). So the "tree" being fit here isn't a tree at all: it's effectively a single linear decision surface, where each word's log-probability acts like a learned coefficient, and whether a word is present or absent (weighted by its TF-IDF value) pushes the total score toward spam or toward ham.

**Contrast with trees**: trees carve space into rectangular regions using many discrete, sequential cuts. Naive Bayes draws one continuous linear boundary using a single weighted vote across every feature simultaneously.

## Extension Questions: Top and Bottom Tokens by P(word | spam)

In [ ]:
# See your hints!
import numpy as np

spam_idx = list(model.classes_).index('spam')
log_probs_spam = model.feature_log_prob_[spam_idx]
probs_spam = np.exp(log_probs_spam)

# Map vocabulary index -> token string
vocab_inv = {index: token for token, index in vectorizer.vocabulary_.items()}

# Sort tokens by P(token | spam), descending
sorted_indices = np.argsort(probs_spam)[::-1]

top_30_indices = sorted_indices[:30]
bottom_30_indices = sorted_indices[-30:][::-1]  # reverse so lowest is last, matching descending display

top_30_tokens = [vocab_inv[i] for i in top_30_indices]
top_30_probs = probs_spam[top_30_indices]

bottom_30_tokens = [vocab_inv[i] for i in bottom_30_indices]
bottom_30_probs = probs_spam[bottom_30_indices]

print("Top 10 tokens by P(token | spam):")
for tok, p in zip(top_30_tokens[:10], top_30_probs[:10]):
    print(f"  {tok}: {p:.6f}")

print("\nBottom 10 tokens by P(token | spam):")
for tok, p in zip(bottom_30_tokens[:10], bottom_30_probs[:10]):
    print(f"  {tok}: {p:.6f}")

In [ ]:
# Plot 1: Top 30 tokens by P(word | spam)
plt.figure(figsize=(14, 5))
plt.bar(top_30_tokens, top_30_probs, color='crimson', alpha=0.8)
plt.xticks(rotation=75, ha='right')
plt.ylabel('P(token | spam)')
plt.title('Top 30 Tokens by P(token | spam)')
plt.tight_layout()
plt.show()

# Plot 2: Bottom 30 tokens by P(word | spam)
plt.figure(figsize=(14, 5))
plt.bar(bottom_30_tokens, bottom_30_probs, color='steelblue', alpha=0.8)
plt.xticks(rotation=75, ha='right')
plt.ylabel('P(token | spam)')
plt.title('Bottom 30 Tokens by P(token | spam)')
plt.tight_layout()
plt.show()

## Question 6 of 7: Patterns in the Plot

**Top 30 tokens** (highest P(word | spam)): dominated by classic marketing/scam vocabulary: "call", "free", "txt", "now", "mobile", "your", plus generic connector words like "to", "or", "for" that happen to appear extremely often in spam's characteristic phrasing ("call now", "txt to claim", "free entry"). These are words that show up in a large fraction of *all* spam messages, so their conditional probability given spam is relatively high.

**Bottom 30 tokens** (lowest P(word | spam)): oddly specific or rare words, many of which look like typos, unusual names, or words that appear in only one or two ham messages in the whole dataset ("nbme", "jstfrnd", "necesity"). They all sit at essentially the same rock-bottom probability, which is the Laplace smoothing floor value, the minimum non-zero probability the model assigns to a word it has seen extremely rarely (or never) in spam messages, so it doesn't assign literal zero probability.

**What makes a token more likely to push toward "spam"?**
- High raw frequency specifically within spam messages (words used repeatedly across many different spam texts, not just once)
- Words tied to urgency, prizes, or calls-to-action ("free", "now", "call", "win")
- Being common enough in general English to appear often, but disproportionately concentrated in the spam class relative to ham

**What makes a token more likely to push toward "ham"?**
- Rare, idiosyncratic words that only showed up in a couple of genuine personal messages (typos, nicknames, in-joke words), these get a near-zero P(word | spam) simply because the model almost never saw them in a spam context, not because they carry some deep "hamminess" signal

This connects back to Question 1's answer about the naive independence assumption: each of these probabilities is estimated completely independently per word, so the model has no way to know that a group of these top words appearing *together* in one message ("free", "call", "now", "mobile" all in one text) is an even stronger spam signal than any one of them alone. It just multiplies their individual contributions.

## Question 7 of 7: Optional File Upload

Optional PDF upload section, not required for this submission. Notebook committed to GitHub in full.